<a href="https://colab.research.google.com/github/mohamdarjoune/Classification/blob/main/PySparl_ML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import pandas as pd
import numpy as np

from pyspark import SparkConf, SparkContext
from pyspark.sql import SparkSession, SQLContext

from pyspark.sql.types import *
import pyspark.sql.functions as F
from pyspark.sql.functions import udf, col

from pyspark.ml.regression import LinearRegression
from pyspark.mllib.evaluation import RegressionMetrics

from pyspark.ml.tuning import ParamGridBuilder, CrossValidator, CrossValidatorModel
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.evaluation import RegressionEvaluator

In [2]:
import seaborn as sns
import matplotlib.pyplot as plt

In [3]:
# Visualization
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [4]:
pd.set_option('display.max_columns', 200)
pd.set_option('display.max_colwidth', 400)

In [5]:
print(pd.set_option)

In [6]:
from matplotlib import rcParams
sns.set(context='notebook', style='whitegrid', rc={'figure.figsize': (18,4)})
rcParams['figure.figsize'] = 18,4

%matplotlib inline
%config InlineBackend.figure_format = 'retina'

In [7]:
# setting random seed for notebook reproducability
rnd_seed=23
np.random.seed=rnd_seed
np.random.set_state=rnd_seed

In [8]:
spark = SparkSession.builder.master("local[2]").appName("Linear-Regression-California-Housing").getOrCreate()

In [9]:
spark

In [10]:
sc = spark.sparkContext
sc

<SparkContext master=local[2] appName=Linear-Regression-California-Housing>

In [11]:
sqlContext = SQLContext(spark.sparkContext)
sqlContext

/usr/local/lib/python3.13/dist-packages/pyspark/sql/context.py:112: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  warnings.warn(


In [22]:
from sklearn.datasets import fetch_california_housing
import pandas as pd

# 1. Charger les données California Housing
housing = fetch_california_housing(as_frame=True)
df = housing.frame

# 2. Réorganiser les colonnes dans l'ordre exact de ton StructType Spark
df_spark = pd.DataFrame({
    'long': df['Longitude'],
    'lat': df['Latitude'],
    'medage': df['HouseAge'],
    'totrooms': df['AveRooms'],
    'totbdrms': df['AveBedrms'],
    'pop': df['Population'],
    'houshlds': df['AveOccup'],
    'medinc': df['MedInc'],
    'medhv': df['MedHouseVal']
})

# 3. Sauvegarder au format texte/CSV SANS en-tête (header=False)
# pour correspondre au fichier .data d'origine
df_spark.to_csv("cal_housing.data", header=False, index=False, sep=',')

print("Fichier cal_housing.data généré avec succès !")

Fichier cal_housing.data généré avec succès !


In [23]:
HOUSING_DATA = "cal_housing.data"
housing_df = spark.read.csv(path=HOUSING_DATA, schema=schema).cache()

In [24]:
# Inspect first five rows
housing_df.take(5)

[Row(long=-122.2300033569336, lat=37.880001068115234, medage=41.0, totrooms=6.984127044677734, totbdrms=1.023809552192688, pop=322.0, houshlds=2.555555582046509, medinc=8.325200080871582, medhv=4.526000022888184),
 Row(long=-122.22000122070312, lat=37.86000061035156, medage=21.0, totrooms=6.238137245178223, totbdrms=0.9718804955482483, pop=2401.0, houshlds=2.109841823577881, medinc=8.301400184631348, medhv=3.5850000381469727),
 Row(long=-122.23999786376953, lat=37.849998474121094, medage=52.0, totrooms=8.288135528564453, totbdrms=1.073446273803711, pop=496.0, houshlds=2.802259922027588, medinc=7.257400035858154, medhv=3.5209999084472656),
 Row(long=-122.25, lat=37.849998474121094, medage=52.0, totrooms=5.817351818084717, totbdrms=1.073059320449829, pop=558.0, houshlds=2.547945261001587, medinc=5.643099784851074, medhv=3.4130001068115234),
 Row(long=-122.25, lat=37.849998474121094, medage=52.0, totrooms=6.281853199005127, totbdrms=1.0810810327529907, pop=565.0, houshlds=2.18146729469299

In [25]:
# Show first five rows
housing_df.show(5)

+-------+-----+------+---------+---------+------+---------+------+-----+
|   long|  lat|medage| totrooms| totbdrms|   pop| houshlds|medinc|medhv|
+-------+-----+------+---------+---------+------+---------+------+-----+
|-122.23|37.88|  41.0| 6.984127|1.0238096| 322.0|2.5555556|8.3252|4.526|
|-122.22|37.86|  21.0|6.2381372|0.9718805|2401.0|2.1098418|8.3014|3.585|
|-122.24|37.85|  52.0| 8.288136|1.0734463| 496.0|  2.80226|7.2574|3.521|
|-122.25|37.85|  52.0| 5.817352|1.0730593| 558.0|2.5479453|5.6431|3.413|
|-122.25|37.85|  52.0| 6.281853| 1.081081| 565.0|2.1814673|3.8462|3.422|
+-------+-----+------+---------+---------+------+---------+------+-----+
only showing top 5 rows


In [26]:
# show the dataframe columns
housing_df.columns

['long',
 'lat',
 'medage',
 'totrooms',
 'totbdrms',
 'pop',
 'houshlds',
 'medinc',
 'medhv']

In [27]:
# show the schema of the dataframe
housing_df.printSchema()

root
 |-- long: float (nullable = true)
 |-- lat: float (nullable = true)
 |-- medage: float (nullable = true)
 |-- totrooms: float (nullable = true)
 |-- totbdrms: float (nullable = true)
 |-- pop: float (nullable = true)
 |-- houshlds: float (nullable = true)
 |-- medinc: float (nullable = true)
 |-- medhv: float (nullable = true)



In [28]:
# run a sample selection
housing_df.select('pop','totbdrms').show(10)

+------+----------+
|   pop|  totbdrms|
+------+----------+
| 322.0| 1.0238096|
|2401.0| 0.9718805|
| 496.0| 1.0734463|
| 558.0| 1.0730593|
| 565.0|  1.081081|
| 413.0|  1.103627|
|1094.0| 0.9513619|
|1157.0| 1.0618238|
|1206.0|  1.117647|
|1551.0|0.99019605|
+------+----------+
only showing top 10 rows
